# Notebook 03 - Detection Models

This notebook trains simple, reproducible fraud-risk detection models using the outputs from Notebook 02.

Expected input files, saved in the same folder as this notebook or in `outputs_02/`:

- `contracts_ie_features.csv`
- `tfidf_matrix.npz`
- `tfidf_vectorizer.joblib`
- `embeddings.npy`
- `contract_ids.csv`

Outputs are saved to `outputs_03/`:

- trained `.joblib` models
- model metrics CSV
- precision-recall curve
- anomaly scores
- error-analysis samples

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Running in:', 'Google Colab' if IN_COLAB else 'Local Jupyter')

if IN_COLAB:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'scikit-learn', 'scipy', 'joblib', 'matplotlib', 'pandas', 'numpy'
    ], check=True)

ROOT = Path.cwd()
INPUT_DIR = ROOT
if not (INPUT_DIR / 'contracts_ie_features.csv').exists() and (ROOT / 'outputs_02' / 'contracts_ie_features.csv').exists():
    INPUT_DIR = ROOT / 'outputs_02'

OUTPUT_DIR = ROOT / 'outputs_03'
OUTPUT_DIR.mkdir(exist_ok=True)

print('INPUT_DIR :', INPUT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from scipy.sparse import load_npz, hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

RANDOM_STATE = 42

## 1. Load Notebook 02 Outputs

In [ ]:
required_files = [
    'contracts_ie_features.csv',
    'tfidf_matrix.npz',
    'tfidf_vectorizer.joblib',
    'embeddings.npy',
    'contract_ids.csv',
]

missing = [name for name in required_files if not (INPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing Notebook 02 output files: {missing}')

df = pd.read_csv(INPUT_DIR / 'contracts_ie_features.csv', low_memory=False)
X_tfidf = load_npz(INPUT_DIR / 'tfidf_matrix.npz')
embeddings = np.load(INPUT_DIR / 'embeddings.npy', mmap_mode='r')
contract_ids = pd.read_csv(INPUT_DIR / 'contract_ids.csv')
tfidf_vectorizer = joblib.load(INPUT_DIR / 'tfidf_vectorizer.joblib')

print(f'df              : {df.shape}')
print(f'TF-IDF matrix   : {X_tfidf.shape}')
print(f'Embeddings      : {embeddings.shape}')
print(f'Contract IDs    : {contract_ids.shape}')

## 2. Validate Row Alignment

In [ ]:
if 'contract_id' not in df.columns:
    raise ValueError('contracts_ie_features.csv must contain contract_id')
if 'contract_id' not in contract_ids.columns:
    raise ValueError('contract_ids.csv must contain contract_id')

if len(df) != X_tfidf.shape[0]:
    raise ValueError(f'Row mismatch: df has {len(df):,} rows, TF-IDF has {X_tfidf.shape[0]:,}')
if len(contract_ids) != embeddings.shape[0]:
    raise ValueError(f'Row mismatch: contract_ids has {len(contract_ids):,} rows, embeddings has {embeddings.shape[0]:,}')

same_order = df['contract_id'].astype(str).reset_index(drop=True).equals(
    contract_ids['contract_id'].astype(str).reset_index(drop=True)
)

if not same_order:
    raise ValueError('contract_ids.csv is not in the same order as contracts_ie_features.csv')

print('All rows are aligned.')

## 3. Build Proxy Target

In [ ]:
# Main target used by the supervised models.
# short_tender_period is excluded by default because OpenTender often marks almost every usable row as flagged.
# Set INCLUDE_SHORT_TENDER = True if the team decides to include it in the final proxy label.
INCLUDE_SHORT_TENDER = False

label_cols = ['single_bid', 'winner_concentration', 'copy_paste_description']
if INCLUDE_SHORT_TENDER:
    label_cols.insert(1, 'short_tender_period')

missing_labels = [col for col in label_cols if col not in df.columns]
if missing_labels:
    raise ValueError(f'Missing label columns: {missing_labels}')

labels = df[label_cols].apply(pd.to_numeric, errors='coerce')
df['target_suspicious'] = labels.eq(1).any(axis=1).astype(int)

print('Target columns:', label_cols)
print(df['target_suspicious'].value_counts().rename(index={0: 'clean', 1: 'suspicious'}))
print(f'Positive rate: {df["target_suspicious"].mean() * 100:.2f}%')

label_summary = []
for col in ['single_bid', 'short_tender_period', 'winner_concentration', 'copy_paste_description', 'shared_address_flag']:
    if col in df.columns:
        s = pd.to_numeric(df[col], errors='coerce')
        label_summary.append({
            'label': col,
            'flagged': int((s == 1).sum()),
            'clean': int((s == 0).sum()),
            'missing': int(s.isna().sum()),
            'flag_rate_known': float((s == 1).sum() / max(((s == 1) | (s == 0)).sum(), 1))
        })

pd.DataFrame(label_summary)

## 4. Structured Features

In [ ]:
structured_cols = [
    'buyer_contracts_count',
    'buyer_total_value',
    'score_integrity',
    'score_transparency',
    'score_tender',
    'ind_single_bid',
    'ind_advertisement_period',
    'ind_procedure_type',
    'ind_call_for_tender',
    'shared_address_flag',
    'num_extracted_companies',
    'num_extracted_locations',
    'has_extracted_company',
    'has_extracted_location',
]
structured_cols = [col for col in structured_cols if col in df.columns]

X_struct_raw = df[structured_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype('float32')

for col in ['buyer_contracts_count', 'buyer_total_value']:
    if col in X_struct_raw.columns:
        X_struct_raw[col] = np.log1p(np.maximum(X_struct_raw[col], 0))

scaler = StandardScaler()
X_struct = scaler.fit_transform(X_struct_raw).astype('float32')

joblib.dump(scaler, OUTPUT_DIR / 'structured_scaler.joblib')
pd.Series(structured_cols, name='feature').to_csv(OUTPUT_DIR / 'structured_feature_columns.csv', index=False)

print(f'Structured features: {X_struct.shape}')
print(structured_cols)

## 5. Train/Test Split

In [ ]:
y = df['target_suspicious'].values.astype(int)
indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

y_train = y[train_idx]
y_test = y[test_idx]

X_train_tfidf = X_tfidf[train_idx]
X_test_tfidf = X_tfidf[test_idx]

X_train_struct = csr_matrix(X_struct[train_idx])
X_test_struct = csr_matrix(X_struct[test_idx])

X_train_tfidf_struct = hstack([X_train_tfidf, X_train_struct], format='csr')
X_test_tfidf_struct = hstack([X_test_tfidf, X_test_struct], format='csr')

X_train_emb = np.asarray(embeddings[train_idx], dtype='float32')
X_test_emb = np.asarray(embeddings[test_idx], dtype='float32')
X_train_emb_struct = np.hstack([X_train_emb, X_struct[train_idx]])
X_test_emb_struct = np.hstack([X_test_emb, X_struct[test_idx]])

print(f'Train rows: {len(train_idx):,}')
print(f'Test rows : {len(test_idx):,}')
print(f'Train positive rate: {y_train.mean() * 100:.2f}%')
print(f'Test positive rate : {y_test.mean() * 100:.2f}%')

## 6. Evaluation Helper

In [ ]:
results = []
predictions = {}

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        'model': name,
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'pr_auc': average_precision_score(y_test, y_prob),
        'roc_auc': roc_auc_score(y_test, y_prob),
    }
    results.append(metrics)
    predictions[name] = {'y_pred': y_pred, 'y_prob': y_prob}

    print('\n' + '=' * 80)
    print(name)
    print('=' * 80)
    print(classification_report(y_test, y_pred, target_names=['Clean', 'Suspicious'], zero_division=0))
    print('Confusion matrix:')
    print(confusion_matrix(y_test, y_pred))
    print(f"PR-AUC: {metrics['pr_auc']:.4f} | ROC-AUC: {metrics['roc_auc']:.4f}")
    return metrics

## 7. Supervised Models

In [ ]:
lr_tfidf = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    solver='saga',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

lr_tfidf.fit(X_train_tfidf, y_train)
joblib.dump(lr_tfidf, OUTPUT_DIR / 'model_logreg_tfidf.joblib')
evaluate_model('Logistic Regression - TF-IDF', lr_tfidf, X_test_tfidf, y_test)

In [ ]:
lr_tfidf_struct = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    solver='saga',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

lr_tfidf_struct.fit(X_train_tfidf_struct, y_train)
joblib.dump(lr_tfidf_struct, OUTPUT_DIR / 'model_logreg_tfidf_structured.joblib')
evaluate_model('Logistic Regression - TF-IDF + Structured', lr_tfidf_struct, X_test_tfidf_struct, y_test)

In [ ]:
lr_emb = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    solver='lbfgs',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

lr_emb.fit(X_train_emb_struct, y_train)
joblib.dump(lr_emb, OUTPUT_DIR / 'model_logreg_embeddings_structured.joblib')
evaluate_model('Logistic Regression - Embeddings + Structured', lr_emb, X_test_emb_struct, y_test)

## 8. Precision-Recall Curves

In [ ]:
plt.figure(figsize=(8, 6))

for name, pred in predictions.items():
    precision, recall, _ = precision_recall_curve(y_test, pred['y_prob'])
    ap = average_precision_score(y_test, pred['y_prob'])
    plt.plot(recall, precision, label=f'{name} (AP={ap:.3f})')

baseline = y_test.mean()
plt.axhline(baseline, linestyle='--', color='gray', label=f'Baseline positive rate ({baseline:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves')
plt.legend(loc='best')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'precision_recall_curves.png', dpi=150)
plt.show()

## 9. Unsupervised Anomaly Detection

In [ ]:
# Isolation Forest is unsupervised. It does not learn the proxy label directly.
# contamination controls the expected share of records flagged as unusual.
CONTAMINATION = 0.10

iso = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

iso.fit(np.asarray(embeddings, dtype='float32'))
iso_raw_pred = iso.predict(np.asarray(embeddings, dtype='float32'))
iso_flag = (iso_raw_pred == -1).astype(int)
anomaly_score = -iso.score_samples(np.asarray(embeddings, dtype='float32'))

joblib.dump(iso, OUTPUT_DIR / 'model_isolation_forest_embeddings.joblib')

anomaly_df = df[['contract_id', 'title', 'buyer_name', 'target_suspicious']].copy()
anomaly_df['anomaly_score'] = anomaly_score
anomaly_df['isolation_forest_flag'] = iso_flag
anomaly_df = anomaly_df.sort_values('anomaly_score', ascending=False)
anomaly_df.to_csv(OUTPUT_DIR / 'anomaly_scores.csv', index=False)

print(f'Isolation Forest flagged {iso_flag.sum():,} contracts ({iso_flag.mean() * 100:.2f}%)')
print(classification_report(y, iso_flag, target_names=['Clean', 'Suspicious'], zero_division=0))

results.append({
    'model': 'Isolation Forest - Embeddings',
    'precision': precision_score(y, iso_flag, zero_division=0),
    'recall': recall_score(y, iso_flag, zero_division=0),
    'f1': f1_score(y, iso_flag, zero_division=0),
    'pr_auc': np.nan,
    'roc_auc': np.nan,
})

anomaly_df.head(10)

## 10. Error Analysis

In [ ]:
results_df = pd.DataFrame(results).sort_values(['pr_auc', 'f1'], ascending=False, na_position='last')
best_supervised_name = results_df.dropna(subset=['pr_auc']).iloc[0]['model']
best_pred = predictions[best_supervised_name]

test_df = df.iloc[test_idx][[
    'contract_id', 'title', 'buyer_name', 'single_bid',
    'winner_concentration', 'copy_paste_description', 'shared_address_flag',
    'num_extracted_companies', 'num_extracted_locations', 'target_suspicious'
]].copy()
if 'short_tender_period' in df.columns:
    test_df['short_tender_period'] = df.iloc[test_idx]['short_tender_period'].values

test_df['predicted_label'] = best_pred['y_pred']
test_df['predicted_probability'] = best_pred['y_prob']

false_positives = test_df[(test_df['target_suspicious'] == 0) & (test_df['predicted_label'] == 1)]
false_negatives = test_df[(test_df['target_suspicious'] == 1) & (test_df['predicted_label'] == 0)]

false_positives.sort_values('predicted_probability', ascending=False).head(25).to_csv(
    OUTPUT_DIR / 'false_positives_sample.csv', index=False
)
false_negatives.sort_values('predicted_probability', ascending=True).head(25).to_csv(
    OUTPUT_DIR / 'false_negatives_sample.csv', index=False
)

print('Best supervised model:', best_supervised_name)
print(f'False positives: {len(false_positives):,}')
print(f'False negatives: {len(false_negatives):,}')

false_negatives.sort_values('predicted_probability').head(10)

## 11. Save Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values(['pr_auc', 'f1'], ascending=False, na_position='last').reset_index(drop=True)
results_df.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

print('Model comparison')
print(results_df.round(4).to_string(index=False))

deliverables = [
    'model_logreg_tfidf.joblib',
    'model_logreg_tfidf_structured.joblib',
    'model_logreg_embeddings_structured.joblib',
    'model_isolation_forest_embeddings.joblib',
    'structured_scaler.joblib',
    'structured_feature_columns.csv',
    'model_comparison.csv',
    'precision_recall_curves.png',
    'anomaly_scores.csv',
    'false_positives_sample.csv',
    'false_negatives_sample.csv',
]

print('\nNotebook 03 deliverables')
print('-' * 70)
for name in deliverables:
    path = OUTPUT_DIR / name
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else 0
    status = 'OK' if path.exists() else 'MISSING'
    print(f'{name:<45} {status:<8} {size_mb:>8.2f} MB')